### New Sentiment Labels
This script went in and retroactively added new sentiment columns to the thread level dataframes to create more meaningful thread level sentiment scores based on individual utterances within them. 

In [1]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np

In [15]:
df_M_convo = pd.read_csv("combined_convo_sent_M.csv")
df_M_utt = pd.read_csv("combined_utt_sent_M.csv")

df_T_convo = pd.read_csv("combined_convo_sent_T.csv")
df_T_utt = pd.read_csv("combined_utt_sent_T.csv")

df_E_convo = pd.read_csv("combined_convo_sent_E.csv")
df_E_utt = pd.read_csv("combined_utt_sent_E.csv")

/var/folders/7t/rp_1dcrj5nv_nqhf5_82lk3h0000gn/T/ipykernel_16292/3911652541.py:2: DtypeWarning: Columns (0: top_level_comment) have mixed types. Specify dtype option on import or set low_memory=False.
  df_M_utt = pd.read_csv("combined_utt_sent_M.csv")
/var/folders/7t/rp_1dcrj5nv_nqhf5_82lk3h0000gn/T/ipykernel_16292/3911652541.py:5: DtypeWarning: Columns (0: top_level_comment) have mixed types. Specify dtype option on import or set low_memory=False.
  df_T_utt = pd.read_csv("combined_utt_sent_T.csv")
/var/folders/7t/rp_1dcrj5nv_nqhf5_82lk3h0000gn/T/ipykernel_16292/3911652541.py:8: DtypeWarning: Columns (0: top_level_comment) have mixed types. Specify dtype option on import or set low_memory=False.
  df_E_utt = pd.read_csv("combined_utt_sent_E.csv")


In [ ]:
# Given VADER compund score, returns the final label in 
# acordance with VADER suggestions
def vader_label(compound):
    if compound >= 0.05:
        return "Positive"
    elif compound <= -0.05:
        return "Negative"
    else:
        return "Neutral"

In [4]:
# roBERTa model doesn't have a compund score like 
# VADER, so this function goes in and tallies up the 
# number of posts and respective sentiments, 
# breaking ties by picking what was originally calculated by 
# roBERTa on the entire text inside of a thread

def bert_majority(row):
        counts = {
            "Negative": row["Neg_bert_tot"],
            "Positive": row["Pos_bert_tot"],
            "Neutral": row["Neut_bert_tot"]
        }

        max_val = max(counts.values())

        winners = [k for k, v in counts.items() if v == max_val]

        if len(winners) == 1:
            return winners[0]

        return row["bert_label"]

In [ ]:
# Retroactively calculating sentiment based on individual posts within
# each thread rather than appending all text and passing that in 
# on its own

def tag_convos(conversation_df, utterances_df):

    conversation_df = conversation_df.copy()
    utterances_df = utterances_df.copy()

    # Calculate average VADER score across the utterances
    # that each thread contains 
    compound_grouping = (utterances_df.groupby("conversation_id")["compound"])
    compound_avg = compound_grouping.mean().reset_index().rename(columns={"compound": "compound_utt"})

    bert_counts = (utterances_df.groupby(["conversation_id", "bert_label"]).size().unstack(fill_value=0).reset_index())

    # Rename what will be the future columns
    bert_counts = bert_counts.rename(columns={
        "Negative": "Neg_bert_tot",
        "Positive": "Pos_bert_tot",
        "Neutral": "Neut_bert_tot"
    })

    conversation_df = conversation_df.merge(compound_avg, on="conversation_id", how="left")
    conversation_df = conversation_df.merge(bert_counts, on="conversation_id", how="left")

    conversation_df["compound_utt"] = conversation_df["compound_utt"].fillna(0)

    conversation_df["compound_utt_label"] = conversation_df["compound_utt"].apply(vader_label)

    conversation_df["bert_majority_label"] = conversation_df.apply(bert_majority, axis=1)

    return conversation_df

In [ ]:
df_M_convo = tag_convos(df_M_convo, df_M_utt)

In [25]:
df_T_convo = tag_convos(df_T_convo, df_T_utt)

In [26]:
df_E_convo = tag_convos(df_E_convo, df_E_utt)

In [ ]:
# import csv

# df_M_convo.to_csv("combined_convo_sent_M_avg.csv", index=False, encoding="utf-8", quoting=csv.QUOTE_ALL)
# df_T_convo.to_csv("combined_convo_sent_T_avg.csv", index=False, encoding="utf-8", quoting=csv.QUOTE_ALL)
# df_E_convo.to_csv("combined_convo_sent_E_avg.csv", index=False, encoding="utf-8", quoting=csv.QUOTE_ALL)